In [1]:
import tensorflow as tf

shakespeare_url = "https://homl.info/shakespeare"  # shortcut URL
filepath = tf.keras.utils.get_file("shakespeare.txt", shakespeare_url)
with open(filepath) as f:
    shakespeare_text = f.read()

In [2]:
print(shakespeare_text[:80])

First Citizen:
Before we proceed any further, hear me speak.

All:
Speak, speak.


以下是处理数据

In [3]:
help(tf.keras.layers.StringLookup())

Help on StringLookup in module keras.src.layers.preprocessing.string_lookup object:

class StringLookup(keras.src.layers.preprocessing.index_lookup.IndexLookup)
 |  StringLookup(max_tokens=None, num_oov_indices=1, mask_token=None, oov_token='[UNK]', vocabulary=None, idf_weights=None, invert=False, output_mode='int', pad_to_max_tokens=False, sparse=False, encoding='utf-8', name=None, **kwargs)
 |  
 |  A preprocessing layer that maps strings to (possibly encoded) indices.
 |  
 |  This layer translates a set of arbitrary strings into integer output via a
 |  table-based vocabulary lookup. This layer will perform no splitting or
 |  transformation of input strings. For a layer that can split and tokenize
 |  natural language, see the `keras.layers.TextVectorization` layer.
 |  
 |  The vocabulary for the layer must be either supplied on construction or
 |  learned via `adapt()`. During `adapt()`, the layer will analyze a data set,
 |  determine the frequency of individual strings tokens,

2026-03-16 16:35:38.726608: I metal_plugin/src/device/metal_device.cc:1154] Metal device set to: Apple M3
2026-03-16 16:35:38.726627: I metal_plugin/src/device/metal_device.cc:296] systemMemory: 16.00 GB
2026-03-16 16:35:38.726631: I metal_plugin/src/device/metal_device.cc:313] maxCacheSize: 5.92 GB
2026-03-16 16:35:38.726644: I tensorflow/core/common_runtime/pluggable_device/pluggable_device_factory.cc:305] Could not identify NUMA node of platform GPU ID 0, defaulting to 0. Your kernel may not have been built with NUMA support.
2026-03-16 16:35:38.726652: I tensorflow/core/common_runtime/pluggable_device/pluggable_device_factory.cc:271] Created TensorFlow device (/job:localhost/replica:0/task:0/device:GPU:0 with 0 MB memory) -> physical PluggableDevice (device: 0, name: METAL, pci bus id: <undefined>)


In [4]:
vocab = sorted(set(shakespeare_text))
#sortesd 确保顺序，  set 找到全部数据中有多少类别，不加sorted就是无序的
char_to_ids = tf.keras.layers.StringLookup(vocabulary=vocab, mask_token=None)
#StringLookup 就是将vocab里边的字符，改成整数ID，就是从1 到 最后一个这样排列，就是给字典里边的每个字编一个号。
#mask_token = none,由于我们是按照这个文本一行一行的读，不需要填充别的，就是none
chars = tf.strings.unicode_split(shakespeare_text, input_encoding="UTF-8")
#unicode_split 将每一个字符分开，而不是整体的单词和句子，变成一个一个的字母和标点和空格等。
encoded_text = char_to_ids(chars)
#这是真正加密的那一行代码，用自己的字典编码，将全部的字符编号，这个时候，encoded_text 就是一个向量了，纯数字，model能看懂的

In [5]:
ids_to_chars = tf.keras.layers.StringLookup(vocabulary=vocab, invert=True, mask_token=None)
#解码器，将数字改成字符的，后续训练的model输出的结果是数字，这个就能将其改为字符

In [6]:
encoded_text[::]

<tf.Tensor: shape=(1115394,), dtype=int64, numpy=array([19, 48, 57, ..., 46,  9,  1])>

In [7]:
"".join(sorted(set(shakespeare_text.lower())))
seq_length = 100

In [8]:
#将原始的向量变成一个一个的数字，就是标量
dataset = tf.data.Dataset.from_tensor_slices(encoded_text)
#将上一步处理好的标量，seq_length+1个一组，跳shift个下一组，
dataset = dataset.window(seq_length + 1, shift=50, drop_remainder=True)
dataset = dataset.flat_map(lambda window: window.batch(seq_length + 1))

In [9]:
dataset

<_FlatMapDataset element_spec=TensorSpec(shape=(None,), dtype=tf.int64, name=None)>

In [10]:
batch_size = 32
#打乱，并分装成32个一组
dataset = dataset.shuffle(10000).batch(batch_size)
#lambda 一次性函数，:左边是输入，右边是输出。 map是批量、自动、高速地对传送带上的每一个数据进行后续的操作
dataset = dataset.map(lambda windows:(windows[:,:-1],windows[:,1:]))
#预处理
dataset = dataset.prefetch(1)

In [11]:
vocab_size = len(vocab) # 词表大小（大概 39 种字符）
embed_size = 16         # 给每个字符分配 16 个维度的“性格特征”

model = tf.keras.Sequential([
    # 1. 把纯数字 ID 变成向量
    tf.keras.layers.Embedding(vocab_size, embed_size, input_shape=[None]),
    
    # 2.第一层 GRU（记忆过去的上下文）
    # return_sequences=True 极其重要！因为我们要它在每个字母后面都吐出一个预测！
    tf.keras.layers.GRU(128, dropout = 0.2, return_sequences=True),
    
    # 3. 第二层 GRU（加深逻辑理解）
    tf.keras.layers.GRU(128, dropout = 0.2, return_sequences=True),
    
    # 4. 输出层
    # 针对65 个字符，打分给出 65 个概率，谁的概率大，下一个字符就是谁！
    tf.keras.layers.Dense(vocab_size, activation="softmax")
])

model.summary()

/opt/miniconda3/envs/tf_env/lib/python3.10/site-packages/keras/src/layers/core/embedding.py:100: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding (Embedding)           │ (None, None, 16)       │         1,040 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ gru (GRU)                       │ (None, None, 128)      │        56,064 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ gru_1 (GRU)                     │ (None, None, 128)      │        99,072 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, None, 65)       │         8,385 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 164,561 (642.82 KB)

 Trainable params: 164,561 (642.82 KB)

 Non-trainable params: 0 (0.00 B)

In [12]:
model.compile(loss="sparse_categorical_crossentropy", 
              optimizer="nadam",
             metrics=["accuracy"])
callback = tf.keras.callbacks.ModelCheckpoint("my_shakespeare_model.keras", monitor="val_accuracy", save_best_only=True)

In [ ]:
#可以自己训练，但是时间太长了，而且不一定好，先用作者训练好的吧
model.fit(dataset,
          epochs = 20,
         callbacks = [callback])

In [13]:
from pathlib import Path

url = "https://github.com/ageron/data/raw/main/shakespeare_model.tgz"
path = tf.keras.utils.get_file("shakespeare_model.tgz", url, extract=True)
if "_extracted" in path:
    model_path = Path(path) / "shakespeare_model"
else:
    model_path = Path(path).with_name("shakespeare_model")



In [14]:
#如果下载一个是压缩包，就是.gtz文件，那就解压缩就好
!tar -xzf shakespeare_model.tgz
#解压缩之后，无论是一个单独的文件，还是一个文件夹，都没有影响!运行下一个code就好

#keras.2 可以运行下一行
shakespeare_model = tf.keras.models.load_model("shakespeare_model")
y_proba = shakespeare_model.predict(["To be or not to b"])[0, -1]
y_pred = tf.argmax(y_proba)  # choose the most probable character ID
text_vec_layer.get_vocabulary()[y_pred + 2]

In [15]:
#keras.3 运行下一行，其表示的意义不一样
model_layer = tf.keras.layers.TFSMLayer("shakespeare_model", call_endpoint='serving_default')

2026-03-16 16:36:08.052461: I tensorflow/core/grappler/optimizers/custom_graph_optimizer_registry.cc:117] Plugin optimizer for device_type GPU is enabled.


In [19]:
text_vec_layer = tf.keras.layers.TextVectorization(
    split="character", 
    standardize="lower"
)
text_vec_layer.adapt([shakespeare_text])
black_box_vocab = text_vec_layer.get_vocabulary()
#这个black_box_vocab就是一个只包含39个字符的字典，

<span style="color:red">下边这个code，可以编写两个函数，来简化一下，一个就是数据的处理，一个就是字符的叠加
但是要注意，在keras.3版本中，下载的model，没有predict（）这个函数</span>

In [20]:
text = "O Romeo, Romeo! wherefore "
temperature = 2.0
for i in range(100):
    
    #直接把当前的一句话塞进黑盒
    X = tf.constant([text])
    output_dict = model_layer(X)
    
    #拿到最后一刻那 39 个字符的概率得分
    raw_tensor = list(output_dict.values())[0]
    last_char_probs = raw_tensor[0, -1, :]  # 取出最后一个字对应的 39 个得分，39个字符出现的概率

    
    # 🚨 战术升级：从“死板的最高分”换成“带温度的轮盘赌”
    # 步骤 A：把概率取对数，还原成原始的得分 (Logits)
    # 加上 1e-7 是为了防止遇到绝对的 0 导致 log(0) 报出 -inf 的错误
    logits = tf.math.log(last_char_probs + 1e-7) 
    # 步骤 B：温度缩放！
    rescaled_logits = logits / temperature
#以上的AB步骤，只是改变了概率的大小，没有改变排序，最大的还是最大的，最小的还是最小的。但是之间的差距变化了，取决于temperature的大小！！
    
    # 步骤 C：categorical  要求输入是二维矩阵 (Batch, 类别数)
    # 我们用 expand_dims 给它套一层纸箱，变成 (1, 39)
    rescaled_logits_2d = tf.expand_dims(rescaled_logits, axis=0)
    
    # 步骤 D：随机抽 1 个，提取它的纯数字 ID。注意⚠️，这里变成了随机抽取一个字符，
    sampled_id = tf.random.categorical(rescaled_logits_2d, num_samples=1)[0, 0].numpy()
#D步骤中，将取最大值的步骤，改成了随机取一个值，但是概率是之前计算的，这个情况就可以发生一定的变化。
    
    # 4. 查字典：把纯数字换成字母，需要+2，因为model里边的设置
    next_char = black_box_vocab[sampled_id+2]
    
    # 5. 拼接到句尾，进入下一次循环
    text += next_char
print(text)
#for循环的劣势，会出现一些无意义的话。

O Romeo, Romeo! wherefore ging
tirded vagabmney?

eetiors
yedd, pexeved
o insuned custio:
an garrscoar betions mote
knower. vo


In [22]:

test = "Good moning,sir,早上好啊，先生"
encode = test.encode("UTF-8")
print(encode)

b'Good moning,sir,\xe6\x97\xa9\xe4\xb8\x8a\xe5\xa5\xbd\xe5\x95\x8a\xef\xbc\x8c\xe5\x85\x88\xe7\x94\x9f'


In [23]:
decode = encode.decode("UTF-8")
print(decode)

Good moning,sir,早上好啊，先生
